<a href="https://colab.research.google.com/github/RobsonLost/ads---faculdade/blob/main/classificacao_de_flores_iris_com_machine_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importação das bibliotecas necessárias
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ============================================================
# PASSO 1: CARREGAR O CONJUNTO DE DADOS IRIS
# ============================================================

# O dataset Iris é um clássico do Machine Learning.
# Ele contém 150 amostras de flores, divididas em 3 espécies (Setosa, Versicolor, Virginica).
# Cada amostra possui 4 características: comprimento/largura da sépala e da pétala.
iris = load_iris()

# X recebe as características (features) - formato: (150, 4)
X = iris.data

# y recebe os rótulos (labels) - 0, 1 ou 2 para cada espécie
y = iris.target

# Criando um DataFrame do Pandas apenas para visualização dos dados (opcional)
df_iris = pd.DataFrame(X, columns=iris.feature_names)
df_iris['especie'] = y
print("📋 Primeiras linhas do DataFrame:")
print(df_iris.head())

# ============================================================
# PASSO 2: PRÉ-PROCESSAMENTO DOS DADOS
# ============================================================

# 2.1 - Dividir os dados em treino (80%) e teste (20%)
# O random_state garante que a divisão seja sempre a mesma, facilitando a reprodutibilidade.
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2.2 - Normalizar os dados com StandardScaler
# A normalização é essencial em redes neurais para que todas as features
# tenham a mesma escala (média 0 e desvio padrão 1), acelerando o treinamento.
scaler = StandardScaler()
X_treino = scaler.fit_transform(X_treino)
X_teste = scaler.transform(X_teste)

print(f"\n✅ Dados divididos: {X_treino.shape[0]} amostras de treino e {X_teste.shape[0]} de teste.")

# ============================================================
# PASSO 3: CONSTRUIR O MODELO DE REDE NEURAL
# ============================================================

# Criando um modelo Sequencial (camadas empilhadas uma após a outra)
modelo = tf.keras.Sequential([
    # Camada de entrada: 4 neurônios (uma para cada característica)
    # Camada oculta: 16 neurônios com ativação ReLU (retorna 0 para negativos, x para positivos)
    tf.keras.layers.Dense(16, activation='relu', input_shape=(4,)),

    # Segunda camada oculta: 8 neurônios com ativação ReLU
    tf.keras.layers.Dense(8, activation='relu'),

    # Camada de saída: 3 neurônios (uma para cada espécie) com ativação Softmax
    # O Softmax transforma os valores em probabilidades que somam 1.
    tf.keras.layers.Dense(3, activation='softmax')
])

# Exibindo a arquitetura do modelo no console
modelo.summary()

# ============================================================
# PASSO 4: COMPILAR E TREINAR O MODELO
# ============================================================

# Compilando o modelo:
# - optimizer='adam': algoritmo de otimização eficiente e muito usado
# - loss='sparse_categorical_crossentropy': função de perda para classificação com múltiplas classes
# - metrics=['accuracy']: queremos acompanhar a acurácia durante o treino
modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Treinando o modelo:
# - epochs=50: o modelo verá o dataset 50 vezes
# - verbose=1: exibe uma barra de progresso no console
historico = modelo.fit(
    X_treino, y_treino,
    epochs=50,
    validation_split=0.2,  # 20% dos dados de treino usados para validação
    verbose=1
)

# ============================================================
# PASSO 5: AVALIAR O MODELO
# ============================================================

# Avaliando o modelo com os dados de teste (que ele nunca viu)
perda, acuracia = modelo.evaluate(X_teste, y_teste, verbose=0)

print(f"\n📊 Resultados da Avaliação:")
print(f"   Perda (Loss): {perda:.4f}")
print(f"   Acurácia     : {acuracia * 100:.2f}%")

# ============================================================
# PASSO 6: FAZER PREVISÕES
# ============================================================

# Fazendo previsões com o modelo treinado
previsoes = modelo.predict(X_teste)

# O modelo retorna probabilidades para cada classe.
# Usamos np.argmax() para pegar a classe com maior probabilidade.
classes_previstas = np.argmax(previsoes, axis=1)

# Exibindo as previsões feitas vs. os valores reais
print("\n🔎 Previsões vs. Valores Reais (primeiras 10 amostras):")
for i in range(10):
    nome_previsto = iris.target_names[classes_previstas[i]]
    nome_real = iris.target_names[y_teste[i]]
    status = "✅" if classes_previstas[i] == y_teste[i] else "❌"
    print(f"   {status} Previsto: {nome_previsto:12s} | Real: {nome_real}")

# ============================================================
# PASSO 7: PREVISÃO DE UMA NOVA FLOR (EXEMPLO PRÁTICO)
# ============================================================

print("\n🌸 Testando o modelo com uma nova flor:")
# Características de uma flor hipotética:
# [comprimento da sépala, largura da sépala, comprimento da pétala, largura da pétala]
nova_flor = np.array([[5.1, 3.5, 1.4, 0.2]])  # Deve ser Setosa
nova_flor_normalizada = scaler.transform(nova_flor)

# Fazendo a previsão
probabilidades = modelo.predict(nova_flor_normalizada, verbose=0)
classe_prevista = np.argmax(probabilidades)
especie_prevista = iris.target_names[classe_prevista]

print(f"   Características: {nova_flor[0]}")
print(f"   Espécie prevista: {especie_prevista}")
print(f"   Probabilidades   : {probabilidades[0]}")

📋 Primeiras linhas do DataFrame:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   especie  
0        0  
1        0  
2        0  
3        0  
4        0  

✅ Dados divididos: 120 amostras de treino e 30 de teste.


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 237ms/step - accuracy: 0.2917 - loss: 1.1593 - val_accuracy: 0.1250 - val_loss: 1.1983
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.3021 - loss: 1.1467 - val_accuracy: 0.1250 - val_loss: 1.1868
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.3021 - loss: 1.1348 - val_accuracy: 0.2500 - val_loss: 1.1757
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.3125 - loss: 1.1229 - val_accuracy: 0.2500 - val_loss: 1.1649
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.3333 - loss: 1.1119 - val_accuracy: 0.2917 - val_loss: 1.1551
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.3542 - loss: 1.1007 - val_accuracy: 0.2917 - val_loss: 1.1457
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.3646 - loss: 1.0901 - val_accuracy: 0.2917 - val_loss: 1.1368
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.4062 - loss: 1.0794 - val_accuracy: 0.3333 - val_loss: 1.